# Dynasty Title Table 文件的处理

historical object schema
- id: uuid
- type:
- name:
- parents: option, [id]
- start_time:
- end_time: option,
- tags

## 1. 朝代处理

In [48]:
%pip install --upgrade numpy pandas openpyxl

240898.85s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


    PyYAML (>=5.1.*)
            ~~~~~~^
Note: you may need to restart the kernel to use updated packages.


In [49]:
# helpers/time_parser.py
import re

def parse_time_string(time_str):
    """
    解析中国古代完整时间字符串
    1. 闰月 → 月份数字为负数 例：闰四月=-4、闰正月=-1
    2. 无年份/月份/日期 → 全部返回 None
    3. 公元前年份=负数，公元后=正数，模糊值(???年)=None
    4. 兼容：初一/初十/二十/廿一/廿九/卅日 全中文日期
    :param time_str: 待解析时间字符串 (支持 916年十二月-922年正月 这类区间格式)
    :return: dict{year:int/None, month:int/None, day:int/None}
    """
    year = None
    month = None
    day = None
    
    # 空值/无意义内容 直接返回全None
    if not time_str or str(time_str).strip() in ["", "???", "未知", "无"]:
        return {"year": year, "month": month, "day": day}
    
    time_str = str(time_str).strip()
    # ===================== 1. 年份解析 =====================
    bc_pattern = r"前(\d+)年"
    ad_pattern = r"(?<!前)(\d+)年"
    bc_match = re.search(bc_pattern, time_str)
    if bc_match:
        year = -int(bc_match.group(1))
    else:
        ad_match = re.search(ad_pattern, time_str)
        if ad_match:
            year = int(ad_match.group(1))

    # ===================== 2. 月份解析【优先级最高，核心修复关键】 =====================
    month_map = {
        '正月': 1, '一月': 1,
        '二月': 2, '三月': 3, '四月': 4, '五月': 5, '六月': 6,
        '七月': 7, '八月': 8, '九月': 9, '十月': 10,
        '十一月': 11, '冬月': 11,
        '十二月': 12, '腊月': 12
    }
    # 正则匹配顺序：先闰月 → 再长月份(十一月/十二月) → 再短月份，彻底避免误匹配
    leap_month_pattern = r"闰(正月|一月|二月|三月|四月|五月|六月|七月|八月|九月|十月|十一月|十二月|冬月|腊月)"
    long_month_pattern = r"(十一月|十二月|冬月|腊月)"
    short_month_pattern = r"(正月|一月|二月|三月|四月|五月|六月|七月|八月|九月|十月)"
    
    leap_month_match = re.search(leap_month_pattern, time_str)
    long_month_match = re.search(long_month_pattern, time_str)
    normal_month_match = re.search(short_month_pattern, time_str)
    
    if leap_month_match:
        month_cn = leap_month_match.group(1)
        month = -month_map[month_cn]  # 闰月=负数
    elif long_month_match:
        month_cn = long_month_match.group(1)
        month = month_map[month_cn]
    elif normal_month_match:
        month_cn = normal_month_match.group(1)
        month = month_map[month_cn]

    # ===================== 3. 日期解析 =====================
    num_map = {'初':0, '十':10, '廿':20, '卅':30,
               '一':1, '二':2, '三':3, '四':4, '五':5, '六':6, '七':7, '八':8, '九':9}
    # 规则1：匹配带月份的日期
    day_pattern = r"(正月|一月|二月|三月|四月|五月|六月|七月|八月|九月|十月|十一月|十二月|冬月|腊月|闰.+?月)(初|十|廿|卅)[一二三四五六七八九]{0,1}日?"
    # 规则2：匹配纯日期
    pure_day_pattern = r"(初|十|廿|卅)[一二三四五六七八九]{0,1}日?"
    day_match = re.search(day_pattern, time_str)
    
    if day_match:
        # 只截取日期部分，剔除前面的月份
        day_cn = day_match.group().split(day_match.group(1))[-1].replace('日','').strip()
        day = 0
        for char in day_cn:
            day += num_map.get(char, 0)
        day = day if day != 0 else 1  # 初一 → 1
    else:
        # 再尝试匹配纯日期（新增逻辑）
        pure_day_match = re.search(pure_day_pattern, time_str)
        if pure_day_match:
            day_cn = pure_day_match.group().replace('日','').strip()
            day = 0
            for char in day_cn:
                day += num_map.get(char, 0)
            day = day if day != 0 else 1  # 初一 → 1

    return {"year": year, "month": month, "day": day}

In [50]:
# helpers/time_parser.py
def parse_time_range_string(time_range_str):
    """
    # 处理中国古代起讫时间字符串，提取起始和终止时间
    # 前1042年-前1021年
    # 前???年-前???年
    # 916年十二月-922年正月
    # 983年六月-1021年闰四月
    # 1086年正月-七月
    # 1129年三月十一日-四月初三
    # 29年闰五月十一日-十七日
    :param time_range_str: 待解析时间字符串 (支持 916年十二月-922年正月 这类区间格式)
    :return: dict{year:int/None, month:int/None, day:int/None}, dict{year:int/None, month:int/None, day:int/None}
    """
    if not isinstance(time_range_str, str):
        return None, None
    
    parts = time_range_str.split('-')
    if len(parts) == 2:
        start_time = parts[0].strip()
        end_time = parts[1].strip()

        start_time = parse_time_string(start_time)
        end_time = parse_time_string(end_time)
        if (end_time['year'] == None and start_time['year'] is not None):
            end_time['year'] = start_time['year']
        if (end_time['month'] == None and end_time['day'] is not None and start_time['month'] is not None):
            end_time['month'] = start_time['month']

        return start_time, end_time
    elif len(parts) == 1 and len(parts[0].strip()) > 0:
        start_time = parts[0].strip()
        start_time = parse_time_string(start_time)

        return start_time, start_time
    else:
        return None, None

In [51]:
# 测试代码
time_range_strs = [
  '前1042年-前1021年',
  '前???年-前???年',
  '916年十二月-922年正月',
  '983年六月-1021年闰四月',
  '1086年正月-七月',
  '1129年三月十一日-四月初三',
  '29年闰五月十一日-十七日',
  '2025年六月'
]

for time_range_str in time_range_strs:
    start_time, end_time = parse_time_range_string(time_range_str)
    print(f"时间字符串：{time_range_str} → 起始时间：{start_time}，终止时间：{end_time}")

时间字符串：前1042年-前1021年 → 起始时间：{'year': -1042, 'month': None, 'day': None}，终止时间：{'year': -1021, 'month': None, 'day': None}
时间字符串：前???年-前???年 → 起始时间：{'year': None, 'month': None, 'day': None}，终止时间：{'year': None, 'month': None, 'day': None}
时间字符串：916年十二月-922年正月 → 起始时间：{'year': 916, 'month': 12, 'day': 12}，终止时间：{'year': 922, 'month': 1, 'day': None}
时间字符串：983年六月-1021年闰四月 → 起始时间：{'year': 983, 'month': 6, 'day': None}，终止时间：{'year': 1021, 'month': -4, 'day': None}
时间字符串：1086年正月-七月 → 起始时间：{'year': 1086, 'month': 1, 'day': None}，终止时间：{'year': 1086, 'month': 7, 'day': None}
时间字符串：1129年三月十一日-四月初三 → 起始时间：{'year': 1129, 'month': 3, 'day': 11}，终止时间：{'year': 1129, 'month': 4, 'day': 3}
时间字符串：29年闰五月十一日-十七日 → 起始时间：{'year': 29, 'month': -5, 'day': 11}，终止时间：{'year': 29, 'month': -5, 'day': 17}
时间字符串：2025年六月 → 起始时间：{'year': 2025, 'month': 6, 'day': None}，终止时间：{'year': 2025, 'month': 6, 'day': None}


In [52]:
import pandas as pd
import json

# 列索引常量定义
USE_COL = 1                 # 是否使用
PERIOD_COL = 2              # 历史分期（分期）
SUB_PERIOD_COL = 3          # 历史分期（时代）
CATEGORY_COL = 4            # 分类
DYNASTY_COL = 5             # 主朝代
POLITY_COL = 6              # 政权
RULER_TITLE_COL = 7         # 谥号
RULER_NAME_COL = 8          # 姓名
RULER_PREVIOUS_COL = 9      # 上一任
RULER_CHANGESTYLE_COL = 10  # 更迭方式
RULER_TIMEINPOWER_COL = 11  # 在位时间
REIGN_TITLE_COL = 12        # 年号
REIGN_TIME_COL = 13         # 年号起讫时间
REIGN_STARTGANZHI_COL = 14  # 年号起始干支
REIGN_STARTNUM_COL = 15     # 年号起始数
REIGN_TIMELENGTH_COL = 16   # 年号使用时间长度


In [53]:
# 从Excel文件中提取历史分期数据的ETL函数
def etl_historical_periods(excel_path, sheet_name, period_type):
    # 1. 读取xlsx文件，指定openpyxl引擎解析xlsx格式
    df = pd.read_excel(
        io=excel_path,
        sheet_name=sheet_name,
        engine="openpyxl",
        dtype=str,                  # 强制所有单元格按字符串读取，避免类型错误
        header=None,                # 【关键1】不把第一行当表头，强制读取所有行（含空行）
        skiprows=3,                 # 【关键2】不跳过任何行
        usecols=None,               # 【关键3】不跳过任何列
        nrows=None,                 # 【关键4】读取全部行，不限行数
        na_filter=False             # 【关键5】不自动过滤空值，保留所有单元格内容
    )

    # 2. 筛选需要的列 + 删除空行，只保留有【大时代】和【子时代】的有效数据
    df_data = df[[period_type, REIGN_TIME_COL]].copy()
    df_data = df_data[
        (df_data[period_type].str.strip()!="")
    ]
    # 重置索引，防止遍历的时候索引错乱
    df_data = df_data.reset_index(drop=True)

    print(f"数据行数：{len(df_data)}")
    print(df_data.head(5))

    # 3. 定义结果列表，按规则提取数据
    result_list = []
    total_rows = len(df_data)

    # 遍历每一行数据
    current_period = None
    for idx in range(total_rows):
        current_row = df_data.iloc[idx]
        period = current_row[period_type]
        regin_time = current_row[REIGN_TIME_COL]

        start_time, _ = parse_time_range_string(str(regin_time).strip())
        if (
            len(str(period).strip()) > 0 and 
            len(result_list) > 0
        ):
                result_list[len(result_list) - 1]["end_time"] = start_time
        
        # 遇到【中华民国】立即终止循环，且不加入结果
        if str(period).strip() == "中华民国":
            break

        # 追加到结果列表
        result_list.append({
            "type": "HISTORICAL_PERIOD",
            "name": str(period).strip(),
            "start_time": start_time,
            "end_time": None,
            "category": 'era_period' if period_type == PERIOD_COL else 'dynasty_period',
            "category:region": 'China',
        })

    return result_list

In [59]:
# 从Excel文件中提取历代年号数据的ETL函数
def etl_historical_regintitle(excel_path, sheet_name):
    # 1. 读取xlsx文件，指定openpyxl引擎解析xlsx格式
    df = pd.read_excel(
        io=excel_path,
        sheet_name=sheet_name,
        engine="openpyxl",
        dtype=str,                  # 强制所有单元格按字符串读取，避免类型错误
        header=None,                # 【关键1】不把第一行当表头，强制读取所有行（含空行）
        skiprows=3,                 # 【关键2】不跳过任何行
        usecols=None,               # 【关键3】不跳过任何列
        nrows=None,                 # 【关键4】读取全部行，不限行数
        na_filter=False             # 【关键5】不自动过滤空值，保留所有单元格内容
    )

    # 2. 筛选需要的列 + 删除空行，只保留有效数据
    df_data = df[[
        USE_COL,
        DYNASTY_COL,
        POLITY_COL,
        RULER_TITLE_COL,
        RULER_NAME_COL,
        RULER_PREVIOUS_COL,
        RULER_CHANGESTYLE_COL,
        RULER_TIMEINPOWER_COL,
        REIGN_TITLE_COL,
        REIGN_TIME_COL,
        REIGN_STARTGANZHI_COL,
        REIGN_STARTNUM_COL,
        REIGN_TIMELENGTH_COL
    ]].copy()
    #df_data = df_data[
    #    (df_data[DYNASTY_COL].str.strip()!="") |
    #    (df_data[SUB_PERIOD_COL].str.strip()!="") |
    #    (df_data[CATEGORY_COL].str.strip()!="") |
    #    (df_data[DYNASTY_COL].str.strip()!="") 
    #]
    # 重置索引，防止遍历的时候索引错乱
    df_data = df_data.reset_index(drop=True)

    print(f"数据行数：{len(df_data)}")
    print(df_data.head(5))

    # 3. 定义结果列表，按规则提取数据
    polity_list = []
    total_rows = len(df_data)

    current_dynasty_name = None
    current_polity = None
    current_rulers = []
    current_regins = []
    # 遍历每一行数据
    for idx in range(total_rows):
        current_row = df_data.iloc[idx]
        dynasty = current_row[DYNASTY_COL]
        polity = current_row[POLITY_COL]
        ruler_title = current_row[RULER_TITLE_COL]
        ruler_name = current_row[RULER_NAME_COL]
        ruler_previous = current_row[RULER_PREVIOUS_COL]
        ruler_changestyle = current_row[RULER_CHANGESTYLE_COL]  # 没有标记的，默认为'继'
        ruler_timeinpower = current_row[RULER_TIMEINPOWER_COL]  # 先不用，通过 regin_time 来计算
        regin_title = current_row[REIGN_TITLE_COL]
        regin_time = current_row[REIGN_TIME_COL]
        regin_startganzhi = current_row[REIGN_STARTGANZHI_COL]  # for check
        regin_startnum = current_row[REIGN_STARTNUM_COL]
        regin_timelength = current_row[REIGN_TIMELENGTH_COL]    # for check
        if (len(str(dynasty).strip()) > 0):
            current_dynasty_name = str(dynasty).strip()
        # 遇到【中华民国】立即终止循环，且不加入结果

        # 新政权
        if (len(str(polity).strip()) > 0):
            if (current_polity is not None):
                current_polity["rulers"] = current_rulers
                current_rulers = []
                polity_list.append(current_polity)

            current_polity_name = str(polity).strip()
            if current_polity_name == "中华民国":
                break

            current_polity = {
                "name": current_polity_name,
                "category:dynasty": current_dynasty_name,
                "category:region": 'China',
            }

        # 新统治者，重置统治者谥号、姓名、在位时间，重置原有年号列表
        if (len(str(ruler_title).strip()) > 0 or len(str(ruler_name).strip()) > 0):
            if (current_polity is not None and current_polity.get("rulers")):
                current_polity["rulers"][len(current_polity["rulers"]) - 1]["regins"] = current_regins
            current_regins = []
            current_ruler_titles = []
            current_ruler_names = []
            current_ruler_previouses = []
            current_ruler_changestyles = []
            current_ruler_start_timeinpower = None
            current_ruler_end_timeinpower = None

            if (len(str(ruler_title).strip()) > 0):
                current_ruler_titles = str(ruler_title).strip().split('、')
            if (len(str(ruler_name).strip()) > 0):
                current_ruler_names = str(ruler_name).strip().split('、')
            if (len(str(ruler_previous).strip()) > 0):
                current_ruler_previouses = str(ruler_previous).strip().split('，')
            if (len(str(ruler_changestyle).strip()) > 0):
                current_ruler_changestyles = str(ruler_changestyle).strip().split('，')
            if (len(str(ruler_timeinpower).strip()) > 0):
                current_ruler_start_timeinpower, current_ruler_end_timeinpower = parse_time_range_string(str(ruler_timeinpower).strip())

            current_rulers.append({
                "title": current_ruler_titles,
                "name": current_ruler_names,
                "previouses": current_ruler_previouses,
                "changestyles": current_ruler_changestyles,
                "start_time": current_ruler_start_timeinpower,
                "end_time": current_ruler_end_timeinpower,
            })

        # 新年号
        if (len(str(regin_title).strip()) > 0):
            current_regin_titles = []
            current_regin_startganzhi = None
            current_regin_startnum = 0
            current_regin_timelength = None
            start_regin_time, end_regin_time = parse_time_range_string(str(regin_time).strip())
            if (len(str(regin_startganzhi).strip()) > 0):
                current_regin_startganzhi = str(regin_startganzhi).strip()
            if (len(str(regin_startnum).strip()) > 0):
                current_regin_startnum = int(str(regin_startnum).strip())
            if (len(str(regin_timelength).strip()) > 0):
                current_regin_timelength = str(regin_timelength).strip()
            current_regin_titles = str(regin_title).strip().split('，')
            current_regins.append({
                "title": current_regin_titles,
                "start_time": start_regin_time,
                "end_time": end_regin_time,
                "time_length": current_regin_timelength,
                "ganzhi": current_regin_startganzhi,
                "number": current_regin_startnum,
            })

    return polity_list

In [55]:
# 转为标准JSON格式（ensure_ascii=False保证中文正常显示，indent美化格式）
def dump_to_json(data, output_path):
    final_json = json.dumps(data, ensure_ascii=False, indent=4)

    # 打印JSON结果
    print("✅ 提取完成，最终JSON结果：")
    print(final_json)

    if (output_path):
        # 可选：将JSON结果保存到本地文件（推荐，方便查看）
        with open(output_path, "w", encoding="utf-8") as f:
            f.write(final_json)
        print(f"\n✅ JSON文件已保存至：{output_path}")

In [60]:
excel_path = '../../docs/Dynasty Title Table20231219.xlsx'
sheet_name = '中国历代年号考'
#data_path = '../../library/meta/'
data_path = '../components/timelinechart/test/data/'
historical_period_path = data_path + 'historical_periods.json'
historical_regintitle_path = data_path + 'historical_regintitles.json'

#era_periods = etl_historical_periods(excel_path, sheet_name, PERIOD_COL)
#dynasty_periods = etl_historical_periods(excel_path, sheet_name, SUB_PERIOD_COL)
#dump_to_json(era_periods + dynasty_periods, historical_period_path)

regintitles = etl_historical_regintitle(excel_path, sheet_name)
dump_to_json(regintitles, historical_regintitle_path)

数据行数：1967
  1  5  6    7   8  9  10             11 12             13 14 15 16
0     周  周  周武王  姬发     建  前1046年-前1043年     前1046年-前1043年         
1           周成王  姬诵        前1042年-前1021年     前1042年-前1021年         
2           周康王  姬钊         前1020年-前996年      前1020年-前996年         
3           周昭王  姬瑕          前995年-前977年       前995年-前977年         
4           周穆王  姬满          前976年-前922年       前976年-前922年         
✅ 提取完成，最终JSON结果：
[
    {
        "name": "周",
        "category:dynasty": "周",
        "category:region": "China",
        "rulers": [
            {
                "title": [
                    "周武王"
                ],
                "name": [
                    "姬发"
                ],
                "previouses": [],
                "changestyles": [
                    "建"
                ],
                "start_time": {
                    "year": -1046,
                    "month": null,
                    "day": null
                },
                "end_time":